In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_2016_2022_bn.csv")

In [3]:
df_se = df_se.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_se

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,DI,COC,CPI,VAA,ROL,EUPC
0,3.01,5.3,5.700000,1.89,24.000000,1.26,21.96,24.326736,26.439038,1.071809,...,78.7,0.07,-76.190000,-483.000000,9.39,2.11,88.0,1.57,1.95,4.791766e+06
1,4.53,5.5,5.800000,1.83,25.000000,1.35,22.16,25.075242,26.261956,0.462320,...,79.2,0.08,2806.440000,-427.000000,9.39,2.10,84.0,1.57,1.82,1.232294e+07
2,-0.13,5.6,5.800000,1.73,25.000000,1.16,22.19,25.308485,26.255213,0.726063,...,79.7,0.07,-972.000000,-2637.000000,9.39,2.11,85.0,1.58,1.78,2.153217e+07
3,2.98,5.3,5.700000,1.69,25.000000,1.01,22.29,24.459560,25.835993,1.515331,...,79.1,0.06,4618.900000,-927.000000,9.39,2.09,85.0,1.56,1.79,1.705523e+07
4,3.40,5.3,5.885385,1.73,25.000000,0.72,21.81,24.410185,26.531166,-2.710996,...,77.5,0.01,6192.000000,-1333.000000,9.26,2.09,85.0,1.50,1.77,1.983904e+07
5,9.10,5.3,5.885385,1.97,26.000000,0.60,23.06,25.256474,25.752442,5.303164,...,77.7,0.05,2098.000000,-1231.357143,9.26,2.10,85.0,1.49,1.70,1.917901e+07
6,9.30,5.3,5.885385,1.86,23.933333,0.68,24.02,26.925817,25.323965,0.771118,...,79.1,0.08,2167.145333,-1231.357143,9.39,2.06,83.0,1.53,1.69,2.225380e+07


In [5]:
discretised_se = pd.DataFrame(index=df_se.index)

for col in df_se.columns:
    no_unique = df_se[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                df_se[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_se[col] = df_se[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_se)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    0    0    0     2     0    2     0     0     2      1  ...    0    1   
1    1    0    0     1     0    2     0     1     1      0  ...    2    2   
2    0    0    0     0     0    1     1     2     1      0  ...    2    1   
3    0    0    0     0     0    1     1     0     0      2  ...    1    0   
4    1    0    1     0     0    0     0     0     2      0  ...    0    0   
5    2    0    1     2     1    0     2     1     0      2  ...    0    0   
6    2    0    1     1     0    0     2     2     0      1  ...    1    2   

   EPFRG  GGAG  DI  COC  CPI  VAA  ROL  EUPC  
0      0     2   0    2    1    1    2     0  
1      1     2   0    1    0    1    2     0  
2      0     0   0    2    0    2    1     2  
3      2     1   0    0    0    1    1     0  
4      2     0   0    0    0    0    0     1  
5      0     0   0    1    0    0    0     1  
6      1     0   0    0    0    0   

In [6]:
sm = from_pandas(discretised_se)

In [7]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_se.html")

Graphs/01_fully_connected_se.html


In [8]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_se.html")

Graphs/01_thresholded_se.html


In [9]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_se_1.html")

In [10]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)

    From     To    Weight
41  PDPS   EUPC  4.240019
30  PDPS   VABI  3.820173
19  EIAP   GGFC  2.944304
56   LTU    LFG  2.887098
12   QOA    EMP  2.812681
..   ...    ...       ...
57   LTU    COC -2.580302
70   CPI    LFG -3.028345
32  PDPS   GGFC -3.101808
39  PDPS  EPFRG -3.423954
71   CPI  EPFRG -3.740031

[78 rows x 3 columns]


In [11]:
df_edges.to_csv("se_2007_2015.csv")